# OmniVoice Fine-tuning on Google Colab

Fine-tune OmniVoice on your own voice dataset (Vietnamese example: `ngoc_huyen_vbee_audiobook`).

**Requirements**: Colab with GPU (T4 free tier works, A100/L4 recommended for speed)

## Pipeline
1. Install OmniVoice
2. Upload/prepare your dataset
3. Extract audio tokens (tokenize audio → WebDataset shards)
4. Run fine-tuning
5. Test the fine-tuned model

## 1. Install OmniVoice

In [ ]:
!pip install -q torch==2.8.0+cu128 torchaudio==2.8.0+cu128 --extra-index-url https://download.pytorch.org/whl/cu128
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q tensorboardX

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## 2. Prepare Dataset

Your dataset needs a JSONL manifest where each line has:
```json
{"audio": "path/to/audio.wav", "text": "transcript text", "language_id": "vi"}
```

Audio should be 24kHz WAV, 3-15 seconds per clip.

In [ ]:
import os

# === OPTION A: Upload from Google Drive ===
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_DIR = "/content/drive/MyDrive/your_dataset"
# TRAIN_JSONL = f"{DATASET_DIR}/train_raw.jsonl"

# === OPTION B: Create a sample dataset for testing ===
DATASET_DIR = "/content/dataset"
TRAIN_JSONL = f"{DATASET_DIR}/train.jsonl"
os.makedirs(f"{DATASET_DIR}/wavs", exist_ok=True)

# Upload your wav files to DATASET_DIR/wavs/ and create the manifest:
# Example manifest creation (replace with your actual data):
print(f"Dataset dir: {DATASET_DIR}")
print(f"Train JSONL: {TRAIN_JSONL}")
print("\nUpload your WAV files to /content/dataset/wavs/")
print("Then create train.jsonl with lines like:")
print('{"audio": "/content/dataset/wavs/001.wav", "text": "your transcript", "language_id": "vi"}')

In [ ]:
# Upload files via Colab UI or from Drive
# Uncomment below to upload from local machine:
# from google.colab import files
# uploaded = files.upload()  # Upload a zip of your dataset
# !unzip -q uploaded_file.zip -d /content/dataset

## 3. Extract Audio Tokens

This converts raw audio into discrete token shards (WebDataset format) using the HiggsAudio tokenizer.

In [ ]:
TOKEN_DIR = "/content/tokens"
os.makedirs(f"{TOKEN_DIR}/train/audios", exist_ok=True)
os.makedirs(f"{TOKEN_DIR}/train/txts", exist_ok=True)

!python -m omnivoice.scripts.extract_audio_tokens \
    --input_jsonl {TRAIN_JSONL} \
    --tar_output_pattern "{TOKEN_DIR}/train/audios/shard-%06d.tar" \
    --jsonl_output_pattern "{TOKEN_DIR}/train/txts/shard-%06d.jsonl" \
    --tokenizer_path eustlb/higgs-audio-v2-tokenizer \
    --nj_per_gpu 3 \
    --loader_workers 4 \
    --shuffle True

In [ ]:
# Verify tokens were extracted
!echo "Shards:" && ls {TOKEN_DIR}/train/audios/ | head -5
!echo "Manifest:" && head -2 {TOKEN_DIR}/train/data.lst

## 4. Configure and Run Fine-tuning

In [ ]:
import json

# Data config
data_config = {
    "train": [{"manifest_path": [f"{TOKEN_DIR}/train/data.lst"]}]
}
DATA_CONFIG = "/content/data_config.json"
with open(DATA_CONFIG, "w") as f:
    json.dump(data_config, f, indent=2)

# Training config
train_config = {
    "llm_name_or_path": "Qwen/Qwen3-0.6B",
    "audio_vocab_size": 1025,
    "audio_mask_id": 1024,
    "num_audio_codebook": 8,
    "audio_codebook_weights": [8, 8, 6, 6, 4, 4, 2, 2],
    "drop_cond_ratio": 0.1,
    "prompt_ratio_range": [0.0, 0.3],
    "mask_ratio_range": [0.0, 1.0],
    "language_ratio": 0.8,
    "use_pinyin_ratio": 0.0,
    "instruct_ratio": 0.0,
    "only_instruct_ratio": 0.0,
    "resume_from_checkpoint": None,
    "init_from_checkpoint": "k2-fsa/OmniVoice",
    "learning_rate": 1e-5,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "steps": 5000,
    "seed": 42,
    "warmup_type": "ratio",
    "warmup_ratio": 0.01,
    "warmup_steps": 0,
    "batch_tokens": 8192,
    "gradient_accumulation_steps": 1,
    "num_workers": 4,
    "mixed_precision": "bf16",
    "allow_tf32": True,
    "attn_implementation": "sdpa",
    "max_sample_tokens": 2000,
    "min_sample_tokens": 50,
    "max_batch_size": 64,
    "logging_steps": 50,
    "eval_steps": 500,
    "save_steps": 1000,
    "keep_last_n_checkpoints": 3
}

TRAIN_CONFIG = "/content/train_config.json"
with open(TRAIN_CONFIG, "w") as f:
    json.dump(train_config, f, indent=2)

OUTPUT_DIR = "/content/exp/omnivoice_finetune"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Training config: {TRAIN_CONFIG}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Steps: {train_config['steps']}, LR: {train_config['learning_rate']}")

In [ ]:
# Launch fine-tuning (single GPU)
!python -m accelerate.commands.launch \
    --gpu_ids 0 \
    --num_processes 1 \
    -m omnivoice.cli.train \
    --train_config {TRAIN_CONFIG} \
    --data_config {DATA_CONFIG} \
    --output_dir {OUTPUT_DIR}

## 5. Test the Fine-tuned Model

In [ ]:
import glob

# Find latest checkpoint
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
CHECKPOINT = checkpoints[-1] if checkpoints else OUTPUT_DIR
print(f"Using checkpoint: {CHECKPOINT}")

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
from IPython.display import Audio

model = OmniVoice.from_pretrained(
    CHECKPOINT,
    device_map="cuda:0",
    dtype=torch.float16
)

# Voice cloning with a reference audio from the dataset
audio = model.generate(
    text="Xin chào, đây là giọng nói được huấn luyện từ dữ liệu của tôi.",
    ref_audio=f"{DATASET_DIR}/wavs/0001_0001.wav",  # use one of your training files as ref
    language="Vietnamese",
)

sf.write("/content/output.wav", audio[0], 24000)
Audio("/content/output.wav")

## 6. Save to Google Drive (Optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r {CHECKPOINT} /content/drive/MyDrive/omnivoice_finetuned/

## Monitor Training (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/tensorboard